### **Дообучение собственной LLM**

##### Проект разделен на две разных части. Первая - pretrain собственной модели. Вторая - posttrain SFT базовой модели `Qwen2.5-0.5B`

In [1]:
import torch
print(torch.__version__)
from torch import cuda


print(cuda.is_available())
print(cuda.get_device_name())

2.6.0+cu124
True
NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
from pathlib import Path


PATH = "RussianNovels-master\corpus"

texts = [
]

for file in Path(PATH).rglob("*.txt"):
    text = file.read_text(encoding="utf-8")
    if len(text) > 100:
        texts.append({
                "path": str(file),
                "text": text
            })
    else: 
        print(f"Пропущен файл: {file}")       
    

In [3]:
texts[0]

{'path': 'RussianNovels-master\\corpus\\Bulgakov_BelayaGvardiya.txt',
 'text': '\n                                 Посвящается Любови Евгеньевне Белозерской\n\n\n\n                             Пошел мелкий снег и вдруг  повалил  хлопьями.\n                          Ветер завыл; сделалась метель. В одно  мгновение\n                          темное  небо  смешалось  с  снежным  морем.  Все\n                          исчезло.\n                             - Ну, барин, - закричал ямщик, - беда: буран!\n                                                       "Капитанская дочка"\n\n                             И судимы были мертвые по написанному в книгах\n                          сообразно с делами своими...\n\n\n\nЧАСТЬ ПЕРВАЯ\n\n\n\n1\n\n\n   Велик был год и страшен год по рождестве Христовом 1918, от  начала  же\nреволюции второй. Был он обилен летом солнцем, а зимою снегом, и  особенно\nвысоко в небе стояли две звезды: звезда пастушеская -  вечерняя  Венера  и\nкрасный, дрожащий Марс.\n

In [4]:
import re

def preprocess_text(text):

    # 1. Удаляем HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # 2. Удаляем повторяющуюся пунктуацию
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    # 3. Нормализуем whitespace
    text = re.sub(r'\s+', ' ', text)

    # 4. Разбиваем на предложения
    sentences = re.split(r'[.!?]+', text)

    filtered = []

    for sentence in sentences:

        sentence = sentence.strip()

        # 5. Только предложения с кириллицей
        if re.search(r'[а-яА-ЯёЁ]', sentence):

            # убираем очень короткий мусор
            if len(sentence.split()) >= 3:
                filtered.append(sentence)

    # 6. Собираем обратно
    text = '. '.join(filtered)

    return text.strip()

In [5]:
filtered = []
counter = 0
unique_texts = set()

for text in texts:

    cleaned_text = preprocess_text(text['text'])

    if cleaned_text not in unique_texts:

        unique_texts.add(cleaned_text)

        filtered.append({
            'path': text['path'],
            'text': cleaned_text
        })

    else:
        counter += 1
        print(f"Пропущен дубликат: {text['path']}")

print(f"Всего пропущено элементов: {counter}")

Пропущен дубликат: RussianNovels-master\corpus\Vovchok_ZapiskiPrichyotnika.txt
Всего пропущено элементов: 1


In [6]:
filtered[0]

{'path': 'RussianNovels-master\\corpus\\Bulgakov_BelayaGvardiya.txt',
 'text': 'Посвящается Любови Евгеньевне Белозерской Пошел мелкий снег и вдруг повалил хлопьями. Ветер завыл; сделалась метель. В одно мгновение темное небо смешалось с снежным морем. - Ну, барин, - закричал ямщик, - беда: буран. "Капитанская дочка" И судимы были мертвые по написанному в книгах сообразно с делами своими. ЧАСТЬ ПЕРВАЯ 1 Велик был год и страшен год по рождестве Христовом 1918, от начала же революции второй. Был он обилен летом солнцем, а зимою снегом, и особенно высоко в небе стояли две звезды: звезда пастушеская - вечерняя Венера и красный, дрожащий Марс. Но дни и в мирные и в кровавые годы летят как стрела, и молодые Турбины не заметили, как в крепком морозе наступил белый, мохнатый декабрь. О, елочный дед наш, сверкающий снегом и счастьем. Мама, светлая королева, где же ты. Через год после того, как дочь Елена повенчалась с капитаном Сергеем Ивановичем Тальбергом, и в ту неделю, когда старший сын, Ал

**Создание и обучение токенизатора**

In [7]:
context_length = 1024
chunk_size = context_length - 2

In [8]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)


In [9]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

In [10]:
tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!")

[('Let', (0, 3)),
 ("'s", (3, 5)),
 ('Ġtest', (5, 10)),
 ('Ġpre', (10, 14)),
 ('-', (14, 15)),
 ('tokenization', (15, 27)),
 ('!', (27, 28))]

In [11]:
def get_training_corpus():
    for i in range(0, len(filtered), 1000):
        batch = filtered[i : i + 1000]

        if batch and isinstance(batch[0], dict):
            yield [row["text"] for row in batch]

        else:
            yield batch

In [12]:

trainer = trainers.BpeTrainer(
    vocab_size=3000,
    special_tokens=[
        "<pad>",
        "<bos>",
        "<eos>",
    ]
)

tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

In [13]:
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

In [14]:
sentence = "Let's test this tokenizer."
encoding = tokenizer.encode(sentence)
start, end = encoding.offsets[4]
sentence[start:end]

's'

In [15]:
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)

['L', 'e', 't', "'", 's', 'Ġ', 't', 'es', 't', 'Ġ', 't', 'h', 'is', 'Ġ', 't', 'o', 'k', 'en', 'i', 'z', 'er', '.']


In [16]:
from transformers import GPT2TokenizerFast
tokenizer.decoder = decoders.ByteLevel()
print(tokenizer.decode(encoding.ids))


wrapped_tokenizer = GPT2TokenizerFast(tokenizer_object=tokenizer)

Let's test this tokenizer.


In [17]:
bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")
pad_id = tokenizer.token_to_id("<pad>")
print(bos_id)

1


Токенизируем весь текст:

In [18]:
all_ids = []

for sample in filtered:
    ids = tokenizer.encode(sample["text"]).ids
    all_ids.extend(ids)

Разделяем текст на чанки:

In [ ]:
def split_text(all_ids, chunk_size):

    chunks = []
    attention_masks = []

    seq_len = chunk_size + 2  # + <bos>, <eos>

    for i in range(0, len(all_ids), chunk_size):
        chunk = all_ids[i : i + chunk_size]

        ids = [bos_id] + chunk + [eos_id]

        pad_len = seq_len - len(ids)
        if pad_len > 0:
            ids = ids + [pad_id] * pad_len

        mask = [1] * (seq_len - max(pad_len, 0)) + [0] * max(pad_len, 0)

        chunks.append(ids)
        attention_masks.append(mask)
    return chunks, attention_masks    

chunks, attention_masks = split_text(all_ids, chunk_size)
chunks[0], attention_masks[0]

([1,
  1552,
  238,
  191,
  820,
  876,
  2266,
  199,
  173,
  2085,
  282,
  360,
  277,
  350,
  455,
  210,
  327,
  208,
  871,
  295,
  1159,
  1922,
  788,
  2828,
  194,
  725,
  602,
  196,
  217,
  2511,
  553,
  642,
  15,
  291,
  218,
  208,
  803,
  742,
  28,
  880,
  635,
  2893,
  330,
  15,
  291,
  1436,
  2107,
  464,
  632,
  477,
  1053,
  169,
  1962,
  598,
  185,
  943,
  373,
  390,
  1478,
  235,
  15,
  213,
  722,
  13,
  1070,
  248,
  13,
  213,
  695,
  2029,
  288,
  195,
  2505,
  13,
  213,
  1333,
  170,
  27,
  2575,
  237,
  15,
  372,
  1855,
  376,
  2852,
  1711,
  1724,
  341,
  3,
  352,
  1470,
  260,
  184,
  714,
  2891,
  440,
  287,
  2256,
  237,
  992,
  188,
  1511,
  433,
  552,
  1379,
  215,
  185,
  506,
  349,
  2424,
  15,
  523,
  1328,
  1776,
  1826,
  127,
  105,
  295,
  2832,
  127,
  166,
  1347,
  1328,
  2285,
  1405,
  291,
  210,
  269,
  306,
  1317,
  194,
  1367,
  203,
  1317,
  287,
  227,
  279,
  468,
  1545,
 

: 

Создаем датасет:

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "input_ids": chunks,
    "attention_mask": attention_masks,
    "labels": chunks,
})

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

In [ ]:
dataset[0]

{'input_ids': [1,
  1552,
  238,
  191,
  820,
  876,
  2266,
  199,
  173,
  2085,
  282,
  360,
  277,
  350,
  455,
  210,
  327,
  208,
  871,
  295,
  1159,
  1922,
  788,
  2828,
  194,
  725,
  602,
  196,
  217,
  2511,
  553,
  642,
  15,
  291,
  218,
  208,
  803,
  742,
  28,
  880,
  635,
  2893,
  330,
  15,
  291,
  1436,
  2107,
  464,
  632,
  477,
  1053,
  169,
  1962,
  598,
  185,
  943,
  373,
  390,
  1478,
  235,
  15,
  213,
  722,
  13,
  1070,
  248,
  13,
  213,
  695,
  2029,
  288,
  195,
  2505,
  13,
  213,
  1333,
  170,
  27,
  2575,
  237,
  15,
  372,
  1855,
  376,
  2852,
  1711,
  1724,
  341,
  3,
  352,
  1470,
  260,
  184,
  714,
  2891,
  440,
  287,
  2256,
  237,
  992,
  188,
  1511,
  433,
  552,
  1379,
  215,
  185,
  506,
  349,
  2424,
  15,
  523,
  1328,
  1776,
  1826,
  127,
  105,
  295,
  2832,
  127,
  166,
  1347,
  1328,
  2285,
  1405,
  291,
  210,
  269,
  306,
  1317,
  194,
  1367,
  203,
  1317,
  287,
  227,
  279,
  4

Создание архитектуры модели:

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
vocab_size = tokenizer.get_vocab_size()
seq_len = context_length
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=seq_len,
    n_ctx=seq_len,
    n_embd=256,
    n_layer=10,
    n_head=8,
    bos_token_id=bos_id,
    eos_token_id=eos_id,
    pad_token_id=pad_id
)
model = GPT2LMHeadModel(config)
model.to("cuda" if torch.cuda.is_available() else "cpu")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0509 18:50:42.549000 26952 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    num_train_epochs=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_steps=500,
    evaluation_strategy="steps",
    eval_steps=500,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
)

trainer.train()

In [ ]:
import math

metrics = trainer.evaluate()
perplexity = math.exp(metrics["eval_loss"]) if "eval_loss" in metrics else None
print({"eval_loss": metrics.get("eval_loss"), "perplexity": perplexity})

### **Оценка модели**

In [ ]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

In [ ]:
for i, test in enumerate(test_prompts):
    tokenized_prompt = tokenizer(test, return_tensors='pt')
    tokenized_prompt = {k: v.to(model.device) for k, v in tokenized_prompt.items()}
    with torch.no_grad():
        outputs = model.generate(
            **tokenized_prompt,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )
    print(f"Пример {i+1}: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")

### **2 этап. Post-train SFT Qwen2.5-0.5B**

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset("d0rj/alpaca-cleaned-ru")

def format_example(example):
    user_text = example["instruction"]
    assistant_text = example["output"]
    if example["input"]:
        system_text = example['input'] 
    else:
        system_text = "You are a helpful assistant."    

    return {
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]
    }

dataset = sft_dataset["train"].map(format_example)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    max_length=512,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    evaluation_strategy="no",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
)

trainer.train()


**Инференс модели**

In [ ]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]

In [ ]:
def format_examples_for_inference(example):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": example},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_ids = inputs['input_ids']
    attention_mask = inputs.get('attention_mask', torch.ones_like(input_ids, dtype=torch.long))

    with torch.no_grad():
        # для воспроизводимости выключим сэмплинг
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False
        )
        # очистим ответ от спецтокенов
        # достаём текст из списка индексом 0
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

for i, question in enumerate(questions_rus):
    answer = format_examples_for_inference(question)
    print(f"Вопрос {i+1}: {question}\nОтвет: {answer}\n")